# Baseline model for batch monitoring example

In [ ]:
import requests
import datetime
import pandas as pd

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric, ColumnQuantileMetric, ColumnCorrelationsMetric

from joblib import load, dump
from tqdm import tqdm

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [ ]:
files = [('green_tripdata_2024-03.parquet', './data')]

print("Download files:")
for file, path in files:
    url=f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file}"
    resp=requests.get(url, stream=True)
    save_path=f"{path}/{file}"
    with open(save_path, "wb") as handle:
        for data in tqdm(resp.iter_content(),
                        desc=f"{file}",
                        postfix=f"save to {save_path}",
                        total=int(resp.headers["Content-Length"])):
            handle.write(data)

In [ ]:
mar_data = pd.read_parquet('data/green_tripdata_2024-03.parquet')

In [ ]:
mar_data.describe()

In [ ]:
mar_data.shape

In [ ]:
# create target
mar_data["duration_min"] = mar_data.lpep_dropoff_datetime - mar_data.lpep_pickup_datetime
mar_data.duration_min = mar_data.duration_min.apply(lambda td : float(td.total_seconds())/60)

In [ ]:
# filter out outliers
mar_data = mar_data[(mar_data.duration_min >= 0) & (mar_data.duration_min <= 60)]
mar_data = mar_data[(mar_data.passenger_count > 0) & (mar_data.passenger_count <= 8)]

In [ ]:
mar_data.duration_min.hist()

In [ ]:
# data labeling
target = "duration_min"
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [ ]:
mar_data.shape

In [ ]:
with open('models/lin_reg.bin', 'rb') as f_in:
    model = load(f_in)

In [ ]:
preds = model.predict(mar_data[num_features + cat_features])
mar_data['prediction'] = preds

In [ ]:
print(mean_absolute_error(mar_data.duration_min, mar_data.prediction))

# Evidently Report

In [ ]:
column_mapping = ColumnMapping(
    target=None,
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features
)

In [ ]:
report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    ColumnQuantileMetric(column_name='fare_amount', quantile=0.5),
    ColumnCorrelationsMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric()
]
)

In [ ]:
reference_data = pd.read_parquet('data/reference.parquet')

In [ ]:
report.run(reference_data=reference_data, current_data=mar_data, column_mapping=column_mapping)

In [ ]:
report.show(mode='inline')

In [ ]:
result = report.as_dict()

In [ ]:
result

In [ ]:
report = Report(metrics=[ColumnQuantileMetric(column_name='fare_amount', quantile=0.5)])
begin = datetime.datetime(2024, 3, 1, 0, 0)
results = []
for d in range(31):
    day = begin + datetime.timedelta(d)
    current_data = mar_data[(mar_data.lpep_pickup_datetime >= day) &
		(mar_data.lpep_pickup_datetime < (day + datetime.timedelta(1)))]
    report.run(reference_data = reference_data, current_data = current_data, column_mapping=column_mapping)
    result = report.as_dict()
    results.append(result['metrics'][0]['result']['current']['value'])
    print(f"{day}: {result['metrics'][0]['result']['current']['value']}")
print(max(results))
    